# Análisis de Trazabilidad y Procesamiento de Logs en Agentes IA

## Objetivo
Aprender a trazar la ejecución completa de un agente de IA, analizar logs estructurados y detectar anomalías en el rendimiento.

---

## ¿Qué es la trazabilidad en sistemas de IA?

En sistemas complejos de IA, una sola solicitud del usuario puede desencadenar múltiples operaciones internas:
validación de entrada, selección de herramientas, llamadas a APIs externas, procesamiento de respuestas, etc.

La **trazabilidad** (tracing) nos permite:

- **Reconstruir el flujo completo** de una solicitud a través de todas sus etapas
- **Identificar cuellos de botella** midiendo la duración de cada sub-operación
- **Diagnosticar errores** sabiendo exactamente dónde y por qué falló una operación
- **Entender relaciones causales** entre componentes del sistema

### Conceptos fundamentales

- **Traza (Trace)**: Representa el recorrido completo de una solicitud. Tiene un `trace_id` único.
- **Span**: Una operación individual dentro de una traza. Tiene inicio, fin, duración y puede tener spans hijos.
- **Span padre/hijo**: Los spans forman un árbol que refleja la jerarquía de operaciones.

```
Traza: "Consulta del usuario"
├── Span: "validar_entrada" (15ms)
├── Span: "seleccionar_herramienta" (5ms)
├── Span: "llamada_api" (1200ms)
│   ├── Span: "construir_prompt" (2ms)
│   ├── Span: "enviar_request" (1150ms)
│   └── Span: "parsear_respuesta" (8ms)
└── Span: "formatear_salida" (10ms)
```

## 0. Configuración del Entorno

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

for env_path in [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path)
        break

if not os.getenv("GITHUB_TOKEN"):
    raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
print("Entorno configurado correctamente")

In [ ]:
# Importaciones generales
import time
import json
import uuid
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Optional
import statistics

import pandas as pd
import matplotlib.pyplot as plt

from openai import OpenAI

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

---
## 1. Generación de Trazas

Construiremos un sistema de trazas con soporte para **spans jerárquicos**, donde cada operación puede contener sub-operaciones.

In [ ]:
@dataclass
class Span:
    """Representa una operación individual dentro de una traza."""
    span_id: str
    nombre: str
    trace_id: str
    parent_span_id: Optional[str] = None
    inicio: Optional[float] = None  # time.perf_counter()
    fin: Optional[float] = None
    duracion_ms: float = 0.0
    estado: str = "EN_PROGRESO"  # EN_PROGRESO, EXITOSO, ERROR
    atributos: dict = field(default_factory=dict)
    error: Optional[str] = None
    timestamp_inicio: Optional[str] = None
    timestamp_fin: Optional[str] = None

    def iniciar(self):
        """Marca el inicio del span."""
        self.inicio = time.perf_counter()
        self.timestamp_inicio = datetime.now(timezone.utc).isoformat()
        return self

    def finalizar(self, estado: str = "EXITOSO", error: str = None):
        """Marca el fin del span y calcula la duración."""
        self.fin = time.perf_counter()
        self.timestamp_fin = datetime.now(timezone.utc).isoformat()
        self.duracion_ms = round((self.fin - self.inicio) * 1000, 2)
        self.estado = estado
        if error:
            self.error = error
        return self


@dataclass
class Traza:
    """Representa el recorrido completo de una solicitud."""
    trace_id: str
    nombre: str
    spans: list = field(default_factory=list)
    timestamp_inicio: Optional[str] = None
    timestamp_fin: Optional[str] = None
    duracion_total_ms: float = 0.0
    estado: str = "EN_PROGRESO"
    metadata: dict = field(default_factory=dict)


class SistemaTrazas:
    """Sistema central de gestión de trazas y spans."""

    def __init__(self):
        self.trazas: list[Traza] = []
        self._traza_activa: Optional[Traza] = None
        self._spans_activos: dict[str, Span] = {}  # span_id -> Span

    def iniciar_traza(self, nombre: str, **metadata) -> Traza:
        """Inicia una nueva traza."""
        traza = Traza(
            trace_id=str(uuid.uuid4())[:8],
            nombre=nombre,
            timestamp_inicio=datetime.now(timezone.utc).isoformat(),
            metadata=metadata
        )
        self.trazas.append(traza)
        self._traza_activa = traza
        return traza

    def iniciar_span(self, nombre: str, parent_span_id: str = None,
                     **atributos) -> Span:
        """Inicia un nuevo span dentro de la traza activa."""
        if not self._traza_activa:
            raise RuntimeError("No hay traza activa. Llama a iniciar_traza() primero.")

        span = Span(
            span_id=str(uuid.uuid4())[:8],
            nombre=nombre,
            trace_id=self._traza_activa.trace_id,
            parent_span_id=parent_span_id,
            atributos=atributos
        )
        span.iniciar()
        self._traza_activa.spans.append(span)
        self._spans_activos[span.span_id] = span
        return span

    def finalizar_span(self, span: Span, estado: str = "EXITOSO",
                       error: str = None, **atributos_extra):
        """Finaliza un span y registra su duración."""
        span.finalizar(estado=estado, error=error)
        span.atributos.update(atributos_extra)
        self._spans_activos.pop(span.span_id, None)

    def finalizar_traza(self, estado: str = "EXITOSO"):
        """Finaliza la traza activa."""
        if self._traza_activa:
            self._traza_activa.timestamp_fin = datetime.now(timezone.utc).isoformat()
            duracion_total = sum(s.duracion_ms for s in self._traza_activa.spans
                                 if s.parent_span_id is None)
            self._traza_activa.duracion_total_ms = round(duracion_total, 2)
            self._traza_activa.estado = estado
            self._traza_activa = None

    def obtener_traza(self, trace_id: str) -> Optional[Traza]:
        """Busca una traza por su ID."""
        for traza in self.trazas:
            if traza.trace_id == trace_id:
                return traza
        return None


# Crear sistema de trazas
sistema_trazas = SistemaTrazas()
print("SistemaTrazas creado exitosamente")

---
## 2. Agente con Trazabilidad Completa

Creamos un agente que registra un span para cada paso de su ejecución: validación de entrada, selección de herramienta, llamada a la API y formateo de respuesta.

In [ ]:
class AgenteTrazable:
    """Agente de IA con trazabilidad completa de cada operación."""

    HERRAMIENTAS_DISPONIBLES = {
        "respuesta_general": "Responde preguntas generales usando el LLM",
        "resumen": "Resume textos largos",
        "traduccion": "Traduce texto entre idiomas",
        "analisis": "Analiza y extrae información de textos"
    }

    def __init__(self, modelo: str = "gpt-4o-mini",
                 sistema_trazas: SistemaTrazas = None):
        self.modelo = modelo
        self.trazas = sistema_trazas or SistemaTrazas()
        self.client = OpenAI(
            base_url=os.getenv("GITHUB_BASE_URL") or os.getenv("OPENAI_BASE_URL", "https://models.inference.ai.azure.com"),
            api_key=os.getenv("GITHUB_TOKEN"),
            timeout=30,
        )
        self.logs: list[dict] = []

    def _log(self, evento: str, trace_id: str, **datos):
        """Registra un log estructurado."""
        registro = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "evento": evento,
            "trace_id": trace_id,
            **datos
        }
        self.logs.append(registro)

    def _validar_entrada(self, mensaje: str, span_padre: str, trace_id: str) -> dict:
        """Span: Validación de la entrada del usuario."""
        span = self.trazas.iniciar_span("validar_entrada",
                                        parent_span_id=span_padre,
                                        largo_mensaje=len(mensaje))
        try:
            errores = []
            if not mensaje or not mensaje.strip():
                errores.append("Mensaje vacío")
            if len(mensaje) > 10000:
                errores.append("Mensaje demasiado largo (>10000 caracteres)")

            es_valido = len(errores) == 0
            self.trazas.finalizar_span(span, es_valido=es_valido,
                                       errores_encontrados=len(errores))
            self._log("validacion_completada", trace_id,
                      es_valido=es_valido, errores=errores)

            return {"valido": es_valido, "errores": errores}

        except Exception as e:
            self.trazas.finalizar_span(span, estado="ERROR", error=str(e))
            raise

    def _seleccionar_herramienta(self, mensaje: str, span_padre: str,
                                  trace_id: str) -> str:
        """Span: Selección de la herramienta apropiada según el mensaje."""
        span = self.trazas.iniciar_span("seleccionar_herramienta",
                                        parent_span_id=span_padre)
        try:
            mensaje_lower = mensaje.lower()
            if any(p in mensaje_lower for p in ["resume", "resumen", "sintetiza"]):
                herramienta = "resumen"
            elif any(p in mensaje_lower for p in ["traduce", "traducción", "translate"]):
                herramienta = "traduccion"
            elif any(p in mensaje_lower for p in ["analiza", "extrae", "identifica"]):
                herramienta = "analisis"
            else:
                herramienta = "respuesta_general"

            self.trazas.finalizar_span(span, herramienta=herramienta)
            self._log("herramienta_seleccionada", trace_id,
                      herramienta=herramienta)

            return herramienta

        except Exception as e:
            self.trazas.finalizar_span(span, estado="ERROR", error=str(e))
            raise

    def _llamar_api(self, mensaje: str, herramienta: str,
                     span_padre: str, trace_id: str) -> dict:
        """Span: Llamada a la API del LLM con sub-spans detallados."""
        span_api = self.trazas.iniciar_span("llamada_api",
                                             parent_span_id=span_padre,
                                             modelo=self.modelo,
                                             herramienta=herramienta)
        try:
            # Sub-span: Construir prompt
            span_prompt = self.trazas.iniciar_span("construir_prompt",
                                                    parent_span_id=span_api.span_id)
            system_prompts = {
                "respuesta_general": "Eres un asistente experto. Responde de forma clara y concisa en español.",
                "resumen": "Eres un experto en sintetizar información. Resume el contenido de forma clara en español.",
                "traduccion": "Eres un traductor profesional. Traduce el texto solicitado manteniendo el significado original.",
                "analisis": "Eres un analista experto. Analiza el contenido y extrae la información clave en español."
            }
            system_prompt = system_prompts.get(herramienta, system_prompts["respuesta_general"])
            mensajes = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": mensaje}
            ]
            self.trazas.finalizar_span(span_prompt, tokens_sistema=len(system_prompt.split()))

            # Sub-span: Enviar request
            span_request = self.trazas.iniciar_span("enviar_request",
                                                     parent_span_id=span_api.span_id,
                                                     modelo=self.modelo)
            respuesta = self.client.chat.completions.create(
                model=self.modelo,
                messages=mensajes,
                temperature=0.7,
                max_tokens=400
            )
            self.trazas.finalizar_span(span_request,
                                       tokens_prompt=respuesta.usage.prompt_tokens,
                                       tokens_completion=respuesta.usage.completion_tokens)

            # Sub-span: Parsear respuesta
            span_parseo = self.trazas.iniciar_span("parsear_respuesta",
                                                    parent_span_id=span_api.span_id)
            contenido = respuesta.choices[0].message.content
            resultado = {
                "contenido": contenido,
                "tokens_prompt": respuesta.usage.prompt_tokens,
                "tokens_completion": respuesta.usage.completion_tokens,
                "tokens_total": respuesta.usage.total_tokens,
                "modelo": self.modelo
            }
            self.trazas.finalizar_span(span_parseo, largo_respuesta=len(contenido))

            # Finalizar span padre de API
            self.trazas.finalizar_span(span_api, tokens_total=resultado["tokens_total"])
            self._log("llamada_api_exitosa", trace_id,
                      tokens_total=resultado["tokens_total"])

            return resultado

        except Exception as e:
            self.trazas.finalizar_span(span_api, estado="ERROR", error=str(e))
            self._log("llamada_api_fallida", trace_id,
                      error=type(e).__name__, mensaje_error=str(e)[:200])
            raise

    def _formatear_salida(self, resultado_api: dict, herramienta: str,
                           span_padre: str, trace_id: str) -> str:
        """Span: Formateo de la respuesta final."""
        span = self.trazas.iniciar_span("formatear_salida",
                                        parent_span_id=span_padre,
                                        herramienta=herramienta)
        try:
            contenido = resultado_api["contenido"]
            respuesta_formateada = contenido.strip()

            self.trazas.finalizar_span(span, largo_salida=len(respuesta_formateada))
            self._log("salida_formateada", trace_id,
                      largo_salida=len(respuesta_formateada))

            return respuesta_formateada

        except Exception as e:
            self.trazas.finalizar_span(span, estado="ERROR", error=str(e))
            raise

    def consultar(self, mensaje: str) -> dict:
        """Ejecuta una consulta completa con trazabilidad de cada paso."""
        # Iniciar traza
        traza = self.trazas.iniciar_traza("consulta_agente", query=mensaje[:100])
        trace_id = traza.trace_id

        # Span raíz
        span_raiz = self.trazas.iniciar_span("procesamiento_completo")
        self._log("consulta_iniciada", trace_id, mensaje=mensaje[:100])

        try:
            # Paso 1: Validar entrada
            validacion = self._validar_entrada(mensaje, span_raiz.span_id, trace_id)
            if not validacion["valido"]:
                self.trazas.finalizar_span(span_raiz, estado="ERROR",
                                           error="Validación fallida")
                self.trazas.finalizar_traza(estado="ERROR")
                return {
                    "trace_id": trace_id,
                    "exitoso": False,
                    "error": f"Validación fallida: {validacion['errores']}"
                }

            # Paso 2: Seleccionar herramienta
            herramienta = self._seleccionar_herramienta(mensaje, span_raiz.span_id, trace_id)

            # Paso 3: Llamar a la API
            resultado_api = self._llamar_api(mensaje, herramienta,
                                              span_raiz.span_id, trace_id)

            # Paso 4: Formatear salida
            respuesta_final = self._formatear_salida(resultado_api, herramienta,
                                                      span_raiz.span_id, trace_id)

            # Finalizar spans y traza
            self.trazas.finalizar_span(span_raiz,
                                       herramienta_usada=herramienta,
                                       tokens_total=resultado_api["tokens_total"])
            self.trazas.finalizar_traza(estado="EXITOSO")

            self._log("consulta_completada", trace_id, exitoso=True)

            return {
                "trace_id": trace_id,
                "exitoso": True,
                "respuesta": respuesta_final,
                "herramienta": herramienta,
                "tokens_total": resultado_api["tokens_total"],
                "duracion_total_ms": traza.duracion_total_ms
            }

        except Exception as e:
            self.trazas.finalizar_span(span_raiz, estado="ERROR", error=str(e))
            self.trazas.finalizar_traza(estado="ERROR")
            self._log("consulta_fallida", trace_id,
                      error=type(e).__name__, mensaje=str(e)[:200])

            return {
                "trace_id": trace_id,
                "exitoso": False,
                "error": str(e)
            }


# Crear el agente trazable
agente = AgenteTrazable(modelo="gpt-4o-mini", sistema_trazas=sistema_trazas)
print("AgenteTrazable creado exitosamente")

In [ ]:
# Prueba básica del agente trazable
resultado = agente.consultar("¿Qué es la inteligencia artificial y por qué es importante?")

print(f"Trace ID: {resultado['trace_id']}")
print(f"Exitoso: {resultado['exitoso']}")
print(f"Herramienta: {resultado.get('herramienta', 'N/A')}")
print(f"Tokens: {resultado.get('tokens_total', 0)}")
print(f"\nRespuesta: {resultado.get('respuesta', resultado.get('error', 'Sin respuesta'))}")

---
## 3. Visualización de Trazas

Vamos a crear funciones para visualizar las trazas de forma comprensible: una vista de **cascada temporal** (waterfall) y un **árbol de spans**.

In [ ]:
def mostrar_cascada(traza: Traza):
    """Muestra una vista de cascada temporal (waterfall) de los spans de una traza."""
    print(f"\n{'='*70}")
    print(f"TRAZA: {traza.nombre} [{traza.trace_id}]")
    print(f"Estado: {traza.estado} | Duración total: {traza.duracion_total_ms}ms")
    print(f"{'='*70}")

    if not traza.spans:
        print("  (sin spans)")
        return

    # Encontrar el tiempo de inicio más temprano para referencia
    inicio_min = min(s.inicio for s in traza.spans if s.inicio is not None)
    duracion_max = max((s.fin or s.inicio) - inicio_min for s in traza.spans
                       if s.inicio is not None) * 1000

    # Ancho de la barra de cascada
    ancho_barra = 40

    print(f"\n{'Span':<25} {'Dur(ms)':>8}  {'Cascada temporal':^{ancho_barra}}")
    print(f"{'─'*25} {'─'*8}  {'─'*ancho_barra}")

    for span in traza.spans:
        if span.inicio is None:
            continue

        # Calcular posición y ancho de la barra
        offset_ms = (span.inicio - inicio_min) * 1000
        if duracion_max > 0:
            pos_inicio = int((offset_ms / duracion_max) * ancho_barra)
            ancho = max(1, int((span.duracion_ms / duracion_max) * ancho_barra))
        else:
            pos_inicio = 0
            ancho = 1

        # Determinar indentación según jerarquía
        nivel = 0
        parent_id = span.parent_span_id
        while parent_id:
            nivel += 1
            parent_span = next((s for s in traza.spans if s.span_id == parent_id), None)
            parent_id = parent_span.parent_span_id if parent_span else None

        indent = "  " * nivel
        nombre_display = f"{indent}{span.nombre}"[:24]

        # Construir la barra visual
        caracter = "#" if span.estado == "EXITOSO" else "X"
        barra = " " * pos_inicio + caracter * ancho
        barra = barra[:ancho_barra].ljust(ancho_barra)

        estado_icon = "OK" if span.estado == "EXITOSO" else "ERR"
        print(f"{nombre_display:<25} {span.duracion_ms:>7.1f}  |{barra}| {estado_icon}")

    print()


def mostrar_arbol(traza: Traza):
    """Muestra un árbol jerárquico de los spans."""
    print(f"\nÁRBOL DE TRAZA [{traza.trace_id}]: {traza.nombre}")
    print(f"{'─'*50}")

    def _imprimir_hijos(parent_id, nivel=0):
        hijos = [s for s in traza.spans if s.parent_span_id == parent_id]
        for i, span in enumerate(hijos):
            es_ultimo = i == len(hijos) - 1
            prefijo = "    " * nivel + ("└── " if es_ultimo else "├── ")
            estado = "[OK]" if span.estado == "EXITOSO" else "[ERR]"
            print(f"{prefijo}{span.nombre} {estado} ({span.duracion_ms}ms)")
            _imprimir_hijos(span.span_id, nivel + 1)

    # Empezar con spans raíz (sin padre)
    raices = [s for s in traza.spans if s.parent_span_id is None]
    for span in raices:
        estado = "[OK]" if span.estado == "EXITOSO" else "[ERR]"
        print(f"{span.nombre} {estado} ({span.duracion_ms}ms)")
        _imprimir_hijos(span.span_id, 1)

    print()


# Visualizar la traza de la consulta anterior
if sistema_trazas.trazas:
    ultima_traza = sistema_trazas.trazas[-1]
    mostrar_cascada(ultima_traza)
    mostrar_arbol(ultima_traza)

In [ ]:
# Ejecutar múltiples consultas para generar más trazas
consultas_variadas = [
    "¿Qué es el aprendizaje profundo?",
    "Resume las ventajas principales de usar Python para ciencia de datos.",
    "Traduce al inglés: La inteligencia artificial está transformando la industria.",
    "Analiza las diferencias entre machine learning y deep learning.",
    "¿Cuáles son los principios SOLID en programación?",
    "Resume el concepto de redes neuronales recurrentes.",
    "¿Qué es el procesamiento de lenguaje natural?",
    "Analiza los beneficios de la computación en la nube.",
    "Traduce al inglés: Los modelos de lenguaje grandes son fundamentales en la IA moderna.",
    "¿Cómo funciona el algoritmo de backpropagation?",
    "Resume las principales aplicaciones de la IA en salud.",
    "Analiza los desafíos éticos de la inteligencia artificial."
]

print(f"Ejecutando {len(consultas_variadas)} consultas con trazabilidad...\n")

resultados = []
for i, consulta in enumerate(consultas_variadas, 1):
    resultado = agente.consultar(consulta)
    resultados.append(resultado)
    estado = "OK" if resultado["exitoso"] else "ERR"
    herramienta = resultado.get("herramienta", "N/A")
    print(f"  [{i:2d}] [{resultado['trace_id']}] [{estado}] {herramienta:<20} | {consulta[:50]}...")

print(f"\nTotal de trazas generadas: {len(sistema_trazas.trazas)}")
print(f"Exitosas: {sum(1 for r in resultados if r['exitoso'])}")
print(f"Fallidas: {sum(1 for r in resultados if not r['exitoso'])}")

In [ ]:
# Visualizar la última traza en detalle
if sistema_trazas.trazas:
    traza_ejemplo = sistema_trazas.trazas[-1]
    mostrar_cascada(traza_ejemplo)
    mostrar_arbol(traza_ejemplo)

---
## 4. Análisis de Logs

Con las trazas generadas, ahora implementamos funciones para analizar los logs y extraer información útil sobre el rendimiento.

In [ ]:
class AnalizadorLogs:
    """Analiza logs y trazas para extraer métricas de rendimiento."""

    def __init__(self, sistema_trazas: SistemaTrazas, logs: list[dict]):
        self.sistema_trazas = sistema_trazas
        self.logs = logs

    def duracion_promedio_por_etapa(self) -> pd.DataFrame:
        """Calcula la duración promedio de cada etapa (tipo de span)."""
        datos = []
        for traza in self.sistema_trazas.trazas:
            for span in traza.spans:
                datos.append({
                    "trace_id": traza.trace_id,
                    "etapa": span.nombre,
                    "duracion_ms": span.duracion_ms,
                    "estado": span.estado
                })

        if not datos:
            return pd.DataFrame()

        df = pd.DataFrame(datos)
        resumen = df.groupby("etapa")["duracion_ms"].agg(
            ["count", "mean", "median", "std", "min", "max"]
        ).round(2)
        resumen.columns = ["Cantidad", "Promedio(ms)", "Mediana(ms)",
                           "Desv.Est(ms)", "Min(ms)", "Max(ms)"]
        return resumen.sort_values("Promedio(ms)", ascending=False)

    def operaciones_mas_lentas(self, top_n: int = 10) -> pd.DataFrame:
        """Identifica las operaciones más lentas a través de todas las trazas."""
        datos = []
        for traza in self.sistema_trazas.trazas:
            for span in traza.spans:
                datos.append({
                    "trace_id": traza.trace_id,
                    "span": span.nombre,
                    "duracion_ms": span.duracion_ms,
                    "estado": span.estado
                })

        if not datos:
            return pd.DataFrame()

        df = pd.DataFrame(datos)
        return df.nlargest(top_n, "duracion_ms").reset_index(drop=True)

    def detectar_anomalias(self, umbral_desviaciones: float = 2.0) -> pd.DataFrame:
        """Detecta operaciones anómalamente lentas (> N desviaciones estándar)."""
        datos = []
        for traza in self.sistema_trazas.trazas:
            for span in traza.spans:
                datos.append({
                    "trace_id": traza.trace_id,
                    "etapa": span.nombre,
                    "duracion_ms": span.duracion_ms
                })

        if not datos:
            return pd.DataFrame()

        df = pd.DataFrame(datos)

        # Calcular estadísticas por etapa
        stats = df.groupby("etapa")["duracion_ms"].agg(["mean", "std"]).reset_index()
        stats.columns = ["etapa", "media", "desv_est"]

        # Merge y filtrar anomalías
        df_merged = df.merge(stats, on="etapa")
        df_merged["z_score"] = ((df_merged["duracion_ms"] - df_merged["media"]) /
                                 df_merged["desv_est"].replace(0, 1)).round(2)
        anomalias = df_merged[df_merged["z_score"].abs() > umbral_desviaciones].copy()

        if anomalias.empty:
            print(f"No se encontraron anomalías con umbral de {umbral_desviaciones} desviaciones estándar.")
        else:
            print(f"Se encontraron {len(anomalias)} operaciones anómalas:")

        return anomalias.sort_values("z_score", ascending=False).reset_index(drop=True)

    def generar_reporte(self) -> str:
        """Genera un reporte textual completo del análisis de trazas."""
        total_trazas = len(self.sistema_trazas.trazas)
        exitosas = sum(1 for t in self.sistema_trazas.trazas if t.estado == "EXITOSO")
        total_spans = sum(len(t.spans) for t in self.sistema_trazas.trazas)
        duraciones = [t.duracion_total_ms for t in self.sistema_trazas.trazas
                      if t.duracion_total_ms > 0]

        reporte = []
        reporte.append("=" * 60)
        reporte.append("REPORTE DE ANÁLISIS DE TRAZABILIDAD")
        reporte.append("=" * 60)
        reporte.append(f"")
        reporte.append(f"Total de trazas: {total_trazas}")
        reporte.append(f"Trazas exitosas: {exitosas} ({exitosas/total_trazas*100:.1f}%)")
        reporte.append(f"Trazas fallidas: {total_trazas - exitosas}")
        reporte.append(f"Total de spans: {total_spans}")
        reporte.append(f"Spans promedio por traza: {total_spans/total_trazas:.1f}")

        if duraciones:
            reporte.append(f"")
            reporte.append(f"--- Rendimiento General ---")
            reporte.append(f"Duración promedio por traza: {statistics.mean(duraciones):.2f}ms")
            reporte.append(f"Duración mediana: {statistics.median(duraciones):.2f}ms")
            if len(duraciones) > 1:
                reporte.append(f"Desviación estándar: {statistics.stdev(duraciones):.2f}ms")
            reporte.append(f"Más rápida: {min(duraciones):.2f}ms")
            reporte.append(f"Más lenta: {max(duraciones):.2f}ms")

        reporte.append(f"")
        reporte.append("=" * 60)

        return "\n".join(reporte)


# Crear analizador
analizador = AnalizadorLogs(sistema_trazas, agente.logs)
print("AnalizadorLogs creado exitosamente")

In [ ]:
# Duración promedio por etapa
print("=== DURACIÓN PROMEDIO POR ETAPA ===")
df_etapas = analizador.duracion_promedio_por_etapa()
print(df_etapas)
print()

In [ ]:
# Operaciones más lentas
print("=== TOP 10 OPERACIONES MÁS LENTAS ===")
df_lentas = analizador.operaciones_mas_lentas(top_n=10)
print(df_lentas)
print()

In [ ]:
# Detección de anomalías
print("=== DETECCIÓN DE ANOMALÍAS ===")
df_anomalias = analizador.detectar_anomalias(umbral_desviaciones=2.0)
if not df_anomalias.empty:
    print(df_anomalias)
print()

In [ ]:
# Reporte completo
reporte = analizador.generar_reporte()
print(reporte)

---
## 5. Detección de Patrones Problemáticos

Analizamos las trazas en conjunto para detectar patrones que indiquen problemas sistémicos: errores recurrentes, degradación de rendimiento y cuellos de botella.

In [ ]:
def analizar_patrones(sistema_trazas: SistemaTrazas):
    """Analiza múltiples trazas para encontrar patrones problemáticos."""

    print("=" * 60)
    print("ANÁLISIS DE PATRONES PROBLEMÁTICOS")
    print("=" * 60)

    trazas = sistema_trazas.trazas
    if len(trazas) < 2:
        print("Se necesitan al menos 2 trazas para análisis de patrones.")
        return

    # 1. Errores recurrentes
    print("\n--- 1. Errores Recurrentes ---")
    errores = {}
    for traza in trazas:
        for span in traza.spans:
            if span.estado == "ERROR" and span.error:
                clave = f"{span.nombre}: {span.error[:80]}"
                errores[clave] = errores.get(clave, 0) + 1

    if errores:
        for error, count in sorted(errores.items(), key=lambda x: x[1], reverse=True):
            print(f"  [{count}x] {error}")
    else:
        print("  No se detectaron errores recurrentes.")

    # 2. Degradación de rendimiento
    print("\n--- 2. Degradación de Rendimiento ---")
    duraciones = [t.duracion_total_ms for t in trazas if t.duracion_total_ms > 0]
    if len(duraciones) >= 4:
        mitad = len(duraciones) // 2
        promedio_primera_mitad = statistics.mean(duraciones[:mitad])
        promedio_segunda_mitad = statistics.mean(duraciones[mitad:])
        cambio_pct = ((promedio_segunda_mitad - promedio_primera_mitad) /
                      promedio_primera_mitad * 100)

        print(f"  Promedio primera mitad: {promedio_primera_mitad:.2f}ms")
        print(f"  Promedio segunda mitad: {promedio_segunda_mitad:.2f}ms")
        print(f"  Cambio: {cambio_pct:+.1f}%")

        if cambio_pct > 20:
            print("  [ALERTA] Se detectó degradación significativa del rendimiento.")
        elif cambio_pct < -20:
            print("  [INFO] Se observa una mejora en el rendimiento.")
        else:
            print("  [OK] Rendimiento estable.")
    else:
        print("  Se necesitan al menos 4 trazas para análisis de tendencia.")

    # 3. Cuellos de botella
    print("\n--- 3. Identificación de Cuellos de Botella ---")
    tiempos_por_etapa = {}
    for traza in trazas:
        for span in traza.spans:
            if span.nombre not in tiempos_por_etapa:
                tiempos_por_etapa[span.nombre] = []
            tiempos_por_etapa[span.nombre].append(span.duracion_ms)

    # Calcular porcentaje de tiempo por etapa
    promedios = {nombre: statistics.mean(tiempos)
                 for nombre, tiempos in tiempos_por_etapa.items()}
    tiempo_total = sum(promedios.values())

    if tiempo_total > 0:
        etapas_ordenadas = sorted(promedios.items(), key=lambda x: x[1], reverse=True)
        for nombre, promedio in etapas_ordenadas[:5]:
            pct = (promedio / tiempo_total) * 100
            barra = "#" * int(pct / 2)
            print(f"  {nombre:<25} {promedio:>8.2f}ms ({pct:>5.1f}%) {barra}")

        cuello = etapas_ordenadas[0]
        print(f"\n  Cuello de botella principal: '{cuello[0]}' ({cuello[1]:.2f}ms promedio)")

    print()


# Ejecutar análisis de patrones
analizar_patrones(sistema_trazas)

In [ ]:
# Visualización gráfica de las trazas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Análisis de Trazabilidad - Agente IA", fontsize=16, fontweight="bold")

# Recopilar datos de spans
datos_spans = []
for traza in sistema_trazas.trazas:
    for span in traza.spans:
        datos_spans.append({
            "trace_id": traza.trace_id,
            "etapa": span.nombre,
            "duracion_ms": span.duracion_ms,
            "estado": span.estado
        })
df_spans = pd.DataFrame(datos_spans)

# 1. Duración promedio por etapa (barras horizontales)
ax1 = axes[0, 0]
etapas_prom = df_spans.groupby("etapa")["duracion_ms"].mean().sort_values()
colores = ["tomato" if v == etapas_prom.max() else "steelblue" for v in etapas_prom.values]
etapas_prom.plot(kind="barh", ax=ax1, color=colores, edgecolor="white")
ax1.set_xlabel("Duración promedio (ms)")
ax1.set_title("Duración Promedio por Etapa")

# 2. Duración total por traza (evolución temporal)
ax2 = axes[0, 1]
duraciones_traza = [t.duracion_total_ms for t in sistema_trazas.trazas if t.duracion_total_ms > 0]
if duraciones_traza:
    ax2.plot(range(1, len(duraciones_traza) + 1), duraciones_traza,
             marker="o", color="steelblue", linewidth=2)
    if len(duraciones_traza) > 1:
        media = statistics.mean(duraciones_traza)
        ax2.axhline(y=media, color="red", linestyle="--",
                    label=f"Promedio: {media:.0f}ms")
        ax2.legend()
ax2.set_xlabel("Número de traza")
ax2.set_ylabel("Duración total (ms)")
ax2.set_title("Evolución de Latencia por Traza")

# 3. Distribución de duración por etapa (boxplot)
ax3 = axes[1, 0]
etapas_principales = ["enviar_request", "llamada_api", "procesamiento_completo"]
df_principales = df_spans[df_spans["etapa"].isin(etapas_principales)]
if not df_principales.empty:
    df_principales.boxplot(column="duracion_ms", by="etapa", ax=ax3)
    ax3.set_xlabel("Etapa")
    ax3.set_ylabel("Duración (ms)")
    ax3.set_title("Distribución de Duración (etapas principales)")
    plt.sca(ax3)
    plt.xticks(rotation=15)
else:
    ax3.text(0.5, 0.5, "Sin datos suficientes", ha="center", va="center")
    ax3.set_title("Distribución de Duración")

# 4. Proporción de tiempo por etapa (pie)
ax4 = axes[1, 1]
# Solo etapas de primer nivel para el pie chart
etapas_nivel1 = df_spans[df_spans["etapa"].isin(
    ["validar_entrada", "seleccionar_herramienta", "llamada_api", "formatear_salida"]
)]
if not etapas_nivel1.empty:
    tiempo_por_etapa = etapas_nivel1.groupby("etapa")["duracion_ms"].sum()
    ax4.pie(tiempo_por_etapa.values, labels=tiempo_por_etapa.index,
            autopct="%1.1f%%", colors=["coral", "steelblue", "mediumseagreen", "goldenrod"])
ax4.set_title("Proporción de Tiempo por Etapa")

# Quitar título automático de boxplot
fig.suptitle("Análisis de Trazabilidad - Agente IA", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Ejercicio Práctico

### Instrucciones

1. **Ejecutar consultas adicionales**: Usa el `agente` para realizar al menos 10 consultas más de distintos tipos (respuesta general, resumen, traducción, análisis)
2. **Visualizar trazas**: Escoge 2-3 trazas y visualízalas con `mostrar_cascada()` y `mostrar_arbol()`
3. **Analizar rendimiento**: Ejecuta el analizador de logs y el detector de patrones
4. **Escribir reporte**: En la celda de reflexión, describe los cuellos de botella encontrados y propuestas de mejora

In [ ]:
# === TU CÓDIGO AQUÍ ===
# Ejecuta al menos 10 consultas adicionales de tipos variados

mis_consultas = [
    # Agrega tus consultas aquí, variando entre tipos:
    # Respuesta general:
    # "¿Qué es Docker y para qué sirve?",
    # Resumen:
    # "Resume los beneficios de la arquitectura de microservicios.",
    # Traducción:
    # "Traduce al inglés: El aprendizaje automático es una rama de la IA.",
    # Análisis:
    # "Analiza las ventajas de usar bases de datos NoSQL.",
]

for consulta in mis_consultas:
    resultado = agente.consultar(consulta)
    print(f"[{resultado['trace_id']}] {resultado.get('herramienta', 'N/A'):<20} - {consulta[:50]}")

In [ ]:
# Visualiza trazas específicas (cambia el índice según tus datos)
# mostrar_cascada(sistema_trazas.trazas[-1])
# mostrar_arbol(sistema_trazas.trazas[-1])

In [ ]:
# Ejecuta el análisis completo
# analizador = AnalizadorLogs(sistema_trazas, agente.logs)
# print(analizador.generar_reporte())
# analizar_patrones(sistema_trazas)

### Reflexión y Reporte

*Escribe aquí tu análisis de rendimiento basado en las trazas:*

- **Cuello de botella principal**: ...
- **Etapa más rápida**: ...
- **Anomalías detectadas**: ...
- **Tendencia de rendimiento**: ...
- **Propuestas de mejora**: ...

---
## Resumen

En este notebook implementamos un sistema completo de trazabilidad:

| Componente | Descripción |
|---|---|
| **Span** | Operación individual con inicio, fin y duración |
| **Traza** | Recorrido completo de una solicitud con múltiples spans |
| **SistemaTrazas** | Gestión centralizada de trazas y spans jerárquicos |
| **AgenteTrazable** | Agente que registra cada paso de su ejecución |
| **Visualización** | Cascada temporal y árbol de spans |
| **AnalizadorLogs** | Análisis estadístico, detección de anomalías y reportes |
| **Patrones** | Identificación de errores recurrentes, degradación y cuellos de botella |

### Conceptos clave
- Las trazas permiten reconstruir el flujo completo de una solicitud
- Los spans jerárquicos revelan dónde se gasta el tiempo en cada operación
- La detección de anomalías (z-score) identifica operaciones fuera de lo normal
- El análisis de tendencias permite detectar degradación progresiva
- Los cuellos de botella casi siempre están en las llamadas a APIs externas
- Herramientas de producción como Jaeger, Zipkin, y OpenTelemetry implementan estos conceptos a escala